$$ VS-Graph \ \  Implementation$$

# Imports

In [ ]:
!pip install  dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html

Looking in links: https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2

In [ ]:
import dgl
from dgl.data import TUDataset
import networkx as nx

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

import time, math, random

from sklearn.model_selection import RepeatedStratifiedKFold

import warnings
warnings.filterwarnings("ignore")


print(torch.__version__)
print(dgl.__version__)

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
2.4.0+cu121
2.4.0+cu124


# Model

In [ ]:
# Full Pipeline + Usage Example
# === Diffusion-Rank → (HD-bipolar | HD-binary | One-Hot rank) → Hebbian GNN ===


# ---------- Quick config ----------
DATASET        = "MUTAG"       # e.g., MUTAG, ENZYMES, DD, NCI1, PROTEINS, PTC_FM
FEAT_TYPE      = "hd_binary"   # one of: "hd_bipolar", "hd_binary", "onehot"
D_HD           = 8192
DIFF_HOPS      = 4
DIFF_NORM      = "sum"         # "sum" or "mean"
DIFF_SELF      = False
DIFF_GSCALE    = None          # None | "l1" | "l2" | "max"

LAYER_MODE     = "or"          # "or" or "sum"
ALPHA          = 0.1
NUM_LAYERS     = 1
MP_SELF        = False

REPEATS        = 3
FOLDS          = 10
SEED           = 0
DEVICE         = "cuda" if __import__("torch").cuda.is_available() else "cpu"
PIN_THREADS    = 1

# ---- Performance toggles (set as you like) ----
USE_FP16            = True      # use float16 on CUDA for features & MP
PACK_BY_NODE_BUDGET = True      # pack batches by total nodes (better GPU utilization)
MAX_NODES_PER_BATCH = 120_000   # target node budget per batch
CACHE_RANKS         = True      # precompute & reuse rank permutations per graph across all folds
COUNT_CACHE_IN_TEST = False     # if True, include cache build time into "inference" timing

# ---------- imports ----------
import os, sys, platform, time, random
import numpy as np
import torch, dgl
from dgl.data import TUDataset
from sklearn.model_selection import RepeatedStratifiedKFold

# ====================== Utilities: env, seeds, timers ======================
def set_all_seeds(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def set_reproducibility(seed=0, num_threads=1, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["OMP_NUM_THREADS"] = str(num_threads)
    os.environ["MKL_NUM_THREADS"] = str(num_threads)
    torch.set_num_threads(num_threads)
    set_all_seeds(seed)
    try: torch.use_deterministic_algorithms(deterministic)
    except: pass
    torch.backends.cudnn.benchmark = False
    try: torch.backends.cudnn.deterministic = deterministic
    except: pass

def report_environment():
    print("="*60)
    print("Environment Summary")
    print("="*60)
    print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")
    if torch.cuda.is_available():
        print("GPU name:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
        try: print("cuDNN version:", torch.backends.cudnn.version())
        except: print("cuDNN version: <unavailable>")
    print("PyTorch:", torch.__version__)
    print("DGL:", dgl.__version__)
    print("Python:", sys.version.split()[0])
    print("Platform:", platform.platform())
    try:
        cpu_line = os.popen("lscpu | grep 'Model name' | head -1").read().strip()
        print("CPU:", cpu_line if cpu_line else "<unavailable>")
    except: print("CPU: <unavailable>")
    print("="*60)

def cuda_sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

class Timer:
    def __init__(self): self.t0 = None
    def start(self): cuda_sync(); self.t0 = time.perf_counter()
    def stop(self):  cuda_sync(); return time.perf_counter() - (self.t0 or time.perf_counter())

def mean_std(arr):
    if not arr: return (0.0, 0.0)
    return float(np.mean(arr)), float(np.std(arr))

def to_numpy_labels(ds) -> np.ndarray:
    ys = [int(y.item()) if isinstance(y, torch.Tensor) else int(y) for _, y in ds]
    return np.asarray(ys, dtype=np.int64)

# ====================== Diffusion (batched) ======================
@torch.no_grad()
def diffusion_score_batched(graphs, hops=3, include_self=False, normalize="sum",
                            global_scale=None, device=torch.device("cpu")):
    if len(graphs) == 0: return []
    gs = [dgl.add_self_loop(g) if include_self else g for g in graphs]
    gs = [g.to(device) if g.device != device else g for g in gs]
    bg = dgl.batch(gs)
    Ns = [int(g.num_nodes()) for g in gs]
    B  = len(gs)
    gid = torch.repeat_interleave(torch.arange(B, device=device), torch.tensor(Ns, device=device))
    v = torch.ones((bg.num_nodes(),), dtype=torch.float32, device=device)
    eps = 1e-12
    for _ in range(hops):
        bg.ndata['h'] = v
        if normalize == "mean":
            bg.update_all(dgl.function.copy_u('h','m'), dgl.function.mean('m','h_new'))
        else:
            bg.update_all(dgl.function.copy_u('h','m'), dgl.function.sum('m','h_new'))
        v = bg.ndata.pop('h_new'); bg.ndata.pop('h', None)
        if global_scale is not None:
            if global_scale == "l1":
                sums = torch.zeros(B, dtype=torch.float32, device=device); sums.scatter_add_(0, gid, v); denom = sums[gid]
            elif global_scale == "l2":
                sq = v * v; sums = torch.zeros(B, dtype=torch.float32, device=device); sums.scatter_add_(0, gid, sq); denom = torch.sqrt(sums[gid] + eps)
            elif global_scale == "max":
                mx = torch.full((B,), -1e38, dtype=torch.float32, device=device)
                mx.scatter_reduce_(0, gid, v, reduce="amax", include_self=True); denom = mx[gid]
            else:
                raise ValueError("global_scale must be None|'l1'|'l2'|'max'")
            v = v / (denom + eps)
    out = []
    start = 0
    for n in Ns:
        out.append(v[start:start+n])
        start += n
    return out

# ====================== DiffusionRank Featurizer (with optional rank cache) ======================
class DiffusionRankFeat:
    """
    Features:
      - "hd_bipolar": D-dim ±1 vector per rank (float in {-1,+1})
      - "hd_binary" : D-dim 0/1 vector per rank (float in {0,1})
      - "onehot"    : global one-hot(rank) of length=max_nodes
    Shared rank-basis across dataset. Optional caching of rank permutations.
    """
    def __init__(self, hops, include_self, normalize, global_scale, feat_type, D, seed, device,
                 use_fp16=False, cache_ranks=False):
        assert feat_type in ("hd_bipolar", "hd_binary", "onehot")
        self.hops, self.include_self, self.normalize, self.global_scale = hops, include_self, normalize, global_scale
        self.feat_type, self.D, self.seed = feat_type, int(D), int(seed)
        self.device = torch.device(device)
        self.rank_basis = None
        self.max_nodes = None
        self.use_fp16 = bool(use_fp16 and torch.cuda.is_available())
        self.cache_ranks = bool(cache_ranks)
        self._rank_cache = []   # stores per-graph inverse-rank (Tensor) aligned with dataset order

    def fit(self, graphs):
        self.max_nodes = max(int(g.num_nodes()) for g in graphs) if graphs else 0
        g_cpu = torch.Generator(device='cpu'); g_cpu.manual_seed(self.seed)
        if self.feat_type == "onehot":
            self.rank_basis = None
        else:
            basis_cpu = torch.empty((self.max_nodes, self.D), dtype=torch.float32)
            basis_cpu.bernoulli_(0.5, generator=g_cpu)
            if self.feat_type == "hd_bipolar":
                basis_cpu = basis_cpu * 2.0 - 1.0
            # hd_binary keeps {0,1}
            dtype = torch.float16 if self.use_fp16 else torch.float32
            self.rank_basis = basis_cpu.to(self.device, dtype=dtype, non_blocking=True)
        # optional rank cache
        if self.cache_ranks:
            self._rank_cache = [None] * len(graphs)
        return self

    @property
    def feature_dim(self):
        return self.max_nodes if self.feat_type == "onehot" else self.D

    @torch.no_grad()
    def _compute_rank_inv(self, graphs, graph_ids=None):
        dev = self.device
        scores = diffusion_score_batched(
            graphs, hops=self.hops, include_self=self.include_self,
            normalize=self.normalize, global_scale=self.global_scale, device=dev
        )
        invs = []
        for sc in scores:
            N = sc.numel()
            ranks_desc = torch.argsort(sc, descending=True)
            inv = torch.empty_like(ranks_desc); inv[ranks_desc] = torch.arange(N, device=sc.device)
            invs.append(inv)
        # update cache if requested
        if self.cache_ranks and graph_ids is not None:
            for inv, gid in zip(invs, graph_ids):
                self._rank_cache[gid] = inv
        return invs

    @torch.no_grad()
    def node_features_batch(self, graphs, graph_ids=None):
        dev = self.device
        F  = self.feature_dim
        if len(graphs) == 0: return []
        # try cache
        use_cache = self.cache_ranks and (graph_ids is not None) and all(self._rank_cache[gid] is not None for gid in graph_ids)
        if use_cache:
            invs = [self._rank_cache[gid] for gid in graph_ids]
        else:
            invs = self._compute_rank_inv(graphs, graph_ids if self.cache_ranks else None)

        feats = []
        for inv in invs:
            N = inv.numel()
            if self.feat_type == "onehot":
                dtype = torch.float16 if self.use_fp16 else torch.float32
                x = torch.zeros((N, F), dtype=dtype, device=dev)
                x[torch.arange(N, device=dev), inv] = 1
            else:
                x = self.rank_basis[inv]
            feats.append(x)
        return feats

# ====================== Hebbian model (fast OR path + FP16 friendly) ======================
class HebbianLayer(torch.nn.Module):
    def __init__(self, mode: str = "or", alpha: float = 0.2, add_self_loop: bool = False):
        super().__init__()
        assert mode in ("or", "sum")
        self.mode = mode
        self.alpha = float(alpha)
        self.add_self_loop = bool(add_self_loop)

    def forward(self, g: dgl.DGLGraph, h: torch.Tensor):
        gg = dgl.add_self_loop(g) if self.add_self_loop else g
        gg = gg.local_var()
        gg.ndata['h'] = h
        if self.mode == "or":

            gg.update_all(dgl.function.copy_u('h','m'), dgl.function.max('m','accum'))
            accum = gg.ndata.pop('accum')
            # For bipolar input, keep sign in {-1,+1}: max over neighbors may yield -1 if all -1; map to ±1
            if torch.min(h) < 0:
                accum = torch.sign(accum)  # still {-1,+1} if inputs are ±1
            else:
                # binary/onehot: ensure {0,1}
                accum = (accum > 0).to(h.dtype)
        else:
            gg.update_all(dgl.function.copy_u('h','m'), dgl.function.sum('m','accum'))
            accum = gg.ndata.pop('accum')
        return self.alpha * h + (1.0 - self.alpha) * accum

class HebbianNet(torch.nn.Module):
    def __init__(self, num_layers=1, mode="or", alpha=0.2, add_self_loop=False):
        super().__init__()
        self.layers = torch.nn.ModuleList(
            [HebbianLayer(mode=mode, alpha=alpha, add_self_loop=add_self_loop) for _ in range(num_layers)]
        )
    def forward(self, g: dgl.DGLGraph, x: torch.Tensor):
        h = x
        for layer in self.layers:
            h = layer(g, h)
        g = g.local_var()
        g.ndata['h'] = h
        hg = dgl.mean_nodes(g, 'h')
        if hg.dim() == 2 and hg.size(0) == 1: return hg.squeeze(0)
        return hg

# ---------- Class mean (cosine) classifier ----------
class ClassMeanClassifier:a
    def __init__(self, feature_dim: int, device: torch.device):
        self.device = device
        self.sum = {}   # lbl -> sum vector (float32)
        self.cnt = {}   # lbl -> count
        self.proto = {} # lbl -> L2-normalized prototype

    def add(self, z: torch.Tensor, y: int):
        # Always store prototypes in float32 for stability
        z = z.to(torch.float32).to(self.device)
        if y not in self.sum:
            self.sum[y] = z.clone()
            self.cnt[y] = 1
        else:
            self.sum[y] += z
            self.cnt[y] += 1

    def finalize(self):
        self.proto = {}
        eps = 1e-12
        for y, s in self.sum.items():
            m = s / max(1, self.cnt[y])
            m = m / (torch.linalg.vector_norm(m, ord=2) + eps)
            self.proto[y] = m

    @torch.no_grad()
    def predict(self, z: torch.Tensor) -> int:
        z = z.to(torch.float32).to(self.device)
        eps = 1e-12
        z = z / (torch.linalg.vector_norm(z, ord=2) + eps)
        best_y, best_dot = None, -1e9
        for y, p in self.proto.items():
            d = torch.dot(z, p).item()
            if d > best_dot:
                best_dot, best_y = d, y
        return best_y

# ====================== batching helpers ======================
def pack_by_node_budget(indices, graphs, max_nodes):
    """Yield lists of indices so that sum(nodes) per batch ≤ max_nodes."""
    if not PACK_BY_NODE_BUDGET:
        yield from chunk_indices(indices, 256)  # fallback chunking by count
        return
    cur, cur_nodes = [], 0
    for idx in indices:
        n = graphs[idx].num_nodes()
        if cur and (cur_nodes + n > max_nodes):
            yield cur
            cur, cur_nodes = [], 0
        cur.append(idx); cur_nodes += n
    if cur: yield cur

def chunk_indices(idxs, size):
    for i in range(0, len(idxs), size):
        yield idxs[i:i+size]

# ====================== Timed CV (unchanged protocol; faster inner loops) ======================
set_reproducibility(seed=SEED, num_threads=PIN_THREADS, deterministic=True)
report_environment()

dev = torch.device(DEVICE)
use_fp16 = bool(USE_FP16 and torch.cuda.is_available())
dtype_feat = torch.float16 if use_fp16 else torch.float32
print(f"Dataset: {DATASET} | Feat={FEAT_TYPE} | D={D_HD} | dtype={str(dtype_feat)} "
      f"| Diffusion(hops={DIFF_HOPS}, norm={DIFF_NORM}, self={DIFF_SELF}, gscale={DIFF_GSCALE}) "
      f"| Hebbian(L={NUM_LAYERS}, mode={LAYER_MODE}, alpha={ALPHA}, self={MP_SELF}) "
      f"| {FOLDS}-fold × {REPEATS} | device={dev} | fp16={use_fp16} | cache_ranks={CACHE_RANKS}")

ds = TUDataset(name=DATASET)
graphs_all = [g.to(dev) for g, _ in ds]
labels_all = to_numpy_labels(ds)
num_classes = int(labels_all.max() + 1)
print(f"#graphs={len(graphs_all)}, #classes={num_classes}")

featurizer = DiffusionRankFeat(
    hops=DIFF_HOPS, include_self=DIFF_SELF, normalize=DIFF_NORM, global_scale=DIFF_GSCALE,
    feat_type=FEAT_TYPE, D=D_HD, seed=SEED, device=dev,
    use_fp16=use_fp16, cache_ranks=CACHE_RANKS
).fit(graphs_all)
F = featurizer.feature_dim
print(f"Feature dim = {F}")

rkf = RepeatedStratifiedKFold(n_splits=FOLDS, n_repeats=REPEATS, random_state=SEED)

accs_per_repeat = [[] for _ in range(REPEATS)]
train_time_per_graph_per_repeat = [[] for _ in range(REPEATS)]
infer_time_per_graph_per_repeat = [[] for _ in range(REPEATS)]
all_fold_accs, all_fold_train_pg, all_fold_infer_pg = [], [], []

# Warmup (helps kernel selection & caches)
def warmup(num=5):
    n = min(len(graphs_all), num)
    if n == 0: return
    ids = list(range(n))
    xs = featurizer.node_features_batch([graphs_all[i] for i in ids], graph_ids=ids)
    bg = dgl.batch([graphs_all[i] for i in ids])
    xcat = torch.cat(xs, dim=0).to(dtype_feat)
    net = HebbianNet(num_layers=NUM_LAYERS, mode=LAYER_MODE, alpha=ALPHA, add_self_loop=MP_SELF).to(dev)
    _ = net(bg, xcat)
warmup()

for k, (train_idx, test_idx) in enumerate(rkf.split(np.zeros_like(labels_all), labels_all)):
    rep = k // FOLDS; fold = k % FOLDS
    net = HebbianNet(num_layers=NUM_LAYERS, mode=LAYER_MODE, alpha=ALPHA, add_self_loop=MP_SELF).to(dev)
    clf = ClassMeanClassifier(feature_dim=F, device=dev)

    # ---- TRAIN timing ----
    t_train = Timer(); t_train.start()
    for batch_ids in pack_by_node_budget(list(train_idx), graphs_all, MAX_NODES_PER_BATCH):
        g_batch = [graphs_all[i] for i in batch_ids]
        feats   = featurizer.node_features_batch(g_batch, graph_ids=batch_ids)
        bg      = dgl.batch(g_batch)
        xcat    = torch.cat(feats, dim=0).to(dtype_feat)
        z_batch = net(bg, xcat)
        if z_batch.dim() == 1: z_batch = z_batch.unsqueeze(0)
        for z_i, i in zip(z_batch, batch_ids):
            clf.add(z_i, int(labels_all[i]))
    clf.finalize()
    train_time_total = t_train.stop()

    # ---- TEST timing ----
    correct, total = 0, 0
    # (Optionally count cache build in test timing)
    build_cache_time = 0.0
    if CACHE_RANKS and COUNT_CACHE_IN_TEST:
        # rebuild ranks into cache here, just for timing (no effect if already cached)
        ids = list(test_idx)
        t_cache = Timer(); t_cache.start()
        _ = featurizer.node_features_batch([graphs_all[i] for i in ids], graph_ids=ids)
        build_cache_time = t_cache.stop()

    t_infer = Timer(); t_infer.start()
    for batch_ids in pack_by_node_budget(list(test_idx), graphs_all, MAX_NODES_PER_BATCH):
        g_batch = [graphs_all[j] for j in batch_ids]
        feats   = featurizer.node_features_batch(g_batch, graph_ids=batch_ids)
        bg      = dgl.batch(g_batch)
        xcat    = torch.cat(feats, dim=0).to(dtype_feat)
        z_batch = net(bg, xcat)
        if z_batch.dim() == 1: z_batch = z_batch.unsqueeze(0)
        for z_j, j in zip(z_batch, batch_ids):
            pred = clf.predict(z_j)
            correct += int(pred == int(labels_all[j])); total += 1
    infer_time_total = t_infer.stop() + (build_cache_time if (CACHE_RANKS and COUNT_CACHE_IN_TEST) else 0.0)

    acc = correct / total if total else 0.0
    accs_per_repeat[rep].append(acc); all_fold_accs.append(acc)
    train_pg = train_time_total / max(1, len(train_idx))
    infer_pg = infer_time_total / max(1, len(test_idx))
    train_time_per_graph_per_repeat[rep].append(train_pg)
    infer_time_per_graph_per_repeat[rep].append(infer_pg)
    all_fold_train_pg.append(train_pg); all_fold_infer_pg.append(infer_pg)

    print(f"[repeat {rep+1}/{REPEATS} | fold {fold+1}/{FOLDS}] "
          f"acc={acc:.4f} | train_total={train_time_total:.6f}s | infer_total={infer_time_total:.6f}s")

    if (fold + 1) == FOLDS:
        r_acc_mean, r_acc_std = mean_std(accs_per_repeat[rep])
        r_tr_mean, r_tr_std   = mean_std(train_time_per_graph_per_repeat[rep])
        r_in_mean, r_in_std   = mean_std(infer_time_per_graph_per_repeat[rep])
        print(f"Repeat {rep+1}: mean acc={r_acc_mean:.4f} (std {r_acc_std:.4f}) | "
              f"train per-graph={r_tr_mean:.6f}s (std {r_tr_std:.6f}) | "
              f"infer per-graph={r_in_mean:.6f}s (std {r_in_std:.6f})")

# ----- Final overall summaries -----
acc_mean, acc_std = mean_std(all_fold_accs)
train_pg_mean, train_pg_std = mean_std(all_fold_train_pg)
infer_pg_mean, infer_pg_std = mean_std(all_fold_infer_pg)
print("="*60)
print("Overall summary across all folds")
print("="*60)
print(f"Accuracy        : mean={acc_mean:.4f}, std={acc_std:.4f}")
print(f"Train per-graph : mean={train_pg_mean:.6f}s, std={train_pg_std:.6f}")
print(f"Infer per-graph : mean={infer_pg_mean:.6f}s, std={infer_pg_std:.6f}")
print("="*60)


Environment Summary
Device: CUDA
GPU name: Tesla T4
CUDA version: 12.1
cuDNN version: 90100
PyTorch: 2.4.0+cu121
DGL: 2.4.0+cu124
Python: 3.12.12
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
CPU: Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
Dataset: MUTAG | Feat=hd_binary | D=8192 | dtype=torch.float16 | Diffusion(hops=4, norm=sum, self=False, gscale=None) | Hebbian(L=1, mode=or, alpha=0.1, self=False) | 10-fold × 3 | device=cuda | fp16=True | cache_ranks=True
#graphs=188, #classes=2
Feature dim = 8192
[repeat 1/3 | fold 1/10] acc=0.9474 | train_total=0.159195s | infer_total=0.025811s
[repeat 1/3 | fold 2/10] acc=0.8421 | train_total=0.020396s | infer_total=0.006102s
[repeat 1/3 | fold 3/10] acc=0.8421 | train_total=0.019948s | infer_total=0.005890s
[repeat 1/3 | fold 4/10] acc=0.8947 | train_total=0.019712s | infer_total=0.005971s
[repeat 1/3 | fold 5/10] acc=0.9474 | train_total=0.019758s | infer_total=0.006236s
[repeat 1/3 | fold 6/10] acc=0.8421 | train